<a href="https://colab.research.google.com/github/rg-smith/remote-sensing-hydro-2026/blob/main/lectures/lecture3-ET.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 3 demo
In this demo, we will load and visualize ET datasets

## Install and import necessary packages

In [ ]:
!pip install geemap

In [17]:
import ee
import folium
import numpy as np
import branca.colormap as cm
import pandas as pd
import zipfile
import os
from tqdm import tqdm
import requests
import geemap

In [3]:
#if not ee.data._credentials:
ee.Authenticate()
ee.Initialize(project='replace with your GEE code')

## Define custom functions

In [18]:
# functions needed for this lab (and some other useful ones that you can use if you're interested)

# to convert a google earth engine image to a python array
def to_array(img,aoi):
  band_arrs = img.sampleRectangle(region=aoi,properties=['scale=1000'],defaultValue=-999)

  band_names=img.bandNames().getInfo()

  for kk in range(len(band_names)):
    if kk==0:
      dat1=np.array(band_arrs.get(band_names[kk]).getInfo())
      dat_full=np.zeros((dat1.shape[0],dat1.shape[1],len(band_names)))
      dat_full[:,:,kk]=dat1
    else:
      dat=np.array(band_arrs.get(band_names[kk]).getInfo())
      dat_full[:,:,kk]=dat
  return(dat_full)

# to calculate an index
def getIndex(image,b1,b2):
  return image.normalizedDifference([b1, b2])

# to calculate a ratio
def getRatio(image1,image2):
  ratio=image1.divide(image2)
  return ratio

# to create a color map from a specific image
def getVisparams(image,aoi):
  range = image.reduceRegion(ee.Reducer.percentile([1, 99]),aoi,300)
  vals = range.getInfo()
  min=list(vals.items())[0][1]
  max=list(vals.items())[1][1]
  visParams = {'min': min, 'max': max, "palette": ["red", "orange", "yellow", "cyan", "blue"]}
  return(visParams)

# to get the link to download an earth engine image
def getLink(image,aoi):
  link = image.getDownloadURL({
    'scale': 1000,
    'crs': 'EPSG:4326',
    'fileFormat': 'GeoTIFF',
    'region': aoi})
  print(link)

# create an earth engine geometry polygon
def addGeometry(min_lon,max_lon,min_lat,max_lat):

  geom = ee.Geometry.Polygon(
      [[[min_lon, max_lat],
        [min_lon, min_lat],
        [max_lon, min_lat],
        [max_lon, max_lat]]])
  return(geom)

def get_center_from_geometry(geom_obj):
  centroid = geom_obj.centroid()
  coords = centroid.getInfo()['coordinates']
  return [coords[1], coords[0]] # Return as [latitude, longitude]

# to export an image to google drive
def export_to_drive(raster,filename,foldername,geometry):
  # Export the image, specifying scale and region.
  task = ee.batch.Export.image.toDrive(**{
      'image': raster,
      'description': filename,
      'folder': foldername,
      'fileNamePrefix': filename,
      'scale': 1000,
      'region': geometry,
      'fileFormat': 'GeoTIFF',
      'formatOptions': {
        'cloudOptimized': 'true'
      },
  })
  task.start()

def get_imgcollection(date1,date2,geometry,collection_name,band_name,function='mean'):
  collection = ee.ImageCollection(collection_name)
  if function=='mean':
      img = collection.filterDate(date1,date2).select(band_name).mean().clip(geometry)
  if function=='sum':
      img = collection.filterDate(date1,date2).select(band_name).sum().clip(geometry)
  return(img)

def get_img(geometry,collection_name,band_name):
  img = ee.Image(collection_name).select(band_name).clip(geometry)
  return(img)

# to get the link to download an earth engine image
def getLink(image,fname,aoi,scale=1000):
  link = image.getDownloadURL({
    'scale': scale,
    'crs': 'EPSG:4326',
    'fileFormat': 'GeoTIFF',
    'region': aoi,
    'name': fname})
  # print(link)
  return(link)

def download_img(img,geom,fname,scale=1000):
    linkname = getLink(img,fname,geom,scale=scale)
    response = requests.get(linkname, stream=True)
    zipped = fname+'.zip'
    with open(zipped, "wb") as handle:
        for data in tqdm(response.iter_content()):
            handle.write(data)

    with zipfile.ZipFile(zipped, 'r') as zip_ref:
        zip_ref.extractall('')
    os.remove(zipped)

## Load and map satellite imagery

First, we define a study area and time period below.

In [11]:
start = 'yyyy-mm-dd'
end = 'yyyy-mm-dd'

geom = addGeometry( ,  , , ) # replace with min long, max long, min lat, max lat of study area
center = get_center_from_geometry(geom)

Now, we will learn to load our own dataset from Google Earth Engine. First, go to the [Earth Engine Catalog](https://https://developers.google.com/earth-engine/datasets), and find a dataset related to ET. Note that you will need to make sure it is available for the time frame and lat/lon that you selected. If it is not, you can change your location/time frame to match.

Once you have found the dataset, copy the path in quotes after the 'Earth Engine Snippet' text. In the code below, the first and second arguments are the start and end date you've already defined. The third is the geometry you've defined. Replace the fourth argument with the path to the image collection you selected. Replace the fifth argument with the band name. Finally, you can choose to take the mean or sum.

In [13]:
img = get_imgcollection(start,end,geom,'replace','replace','sum')
vis = getVisparams(img,geom) # create visualization for ET
layer_name = 'replace with layer name'

Now we will display a map with the image you loaded. Make sure the map is centered on the same region you defined as your study area.

In [ ]:
Map = geemap.Map(center=center, zoom=9)
Map.add_basemap("HYBRID")

Map.addLayer(img,vis,layer_name)
Map.add_colorbar(vis,label='Replace with label',layer_name=layer_name)

Map.addLayer(geom, {},"Study area")

Map.add_inspector()

Map #visualize the map

Now iterate on the code above to add more layers to your map. Explore the Earth Engine Catalog to find layers of interest.

## Download imagery
Now we will download one of the images.

In [ ]:
download_img(img,geom,'output_filename',scale=100)